# Make figure panels from your neighbourhood clusters

The previous notebooks produced **shape clusters** (what a cell looks like) and
**niche clusters** (what the cells around it look like). This notebook turns
both into figure panels: SVG (vector, editable) and 300 dpi PNG, in the
`figures/` folder.

What you do:

1. Point the notebook at `shape_embedding.csv` and `niche_embedding.csv` (one cell below).
2. Run the cells from top to bottom.

Panels in this notebook:

| Section | Panel |
|---|---|
| 3 | The niche map coloured by niche cluster |
| 4 | The shape map and the niche map, each coloured by shape cluster and by niche cluster |
| 5 | Maps coloured by a measurement |
| 6 | One violin per niche cluster, for chosen measurements |
| 7 | Bars: which clusters make up each group (any column) |
| 8 | Which shape clusters live in which niches (log2 observed / expected) |
| 9 | Pictures of typical cells of each niche (optional) |
| 10 | Do replicate stacks mix in the embeddings? (only when the table has several stacks) |

3D viewing of the clusters in napari is planned for a later notebook.

In [ ]:
# Make the helper file next to this notebook importable (works from the repo root or from this folder).
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "tutorials" / "analysis", Path.cwd().parent):
    if (_p / "_helpers.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find _helpers.py. Open this notebook from the repository root or from tutorials/analysis.")

import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _helpers as h

warnings.filterwarnings("ignore", category=FutureWarning)
h.apply_style()
print("Ready.")

## 1. Edit this cell

- `NICHE_EMBEDDING_CSV`: the file written by the neighbourhood notebook. It already holds the shape columns.
- `SHAPE_EMBEDDING_CSV`: the file from the first notebook. Only used to fill in columns that are missing from the niche file. `None` skips it.
- `METADATA_CSV`: optional. One row per stack: a `stack_name` column plus any columns you like. `None` if you have none.
- `SHAPE_RESOLUTION`, `NICHE_RESOLUTION`: which cluster resolutions to show. They must exist in the files (`leiden_r<resolution>` and `niche_leiden_r<resolution>`).
- `GROUP_COL`: a column that defines groups for the bars (section 7). `None` uses `stack_name`.
- `COLOR_FEATURES`, `VIOLIN_FEATURES`: measurements to colour the maps by (section 5) and to draw as violins (section 6). `None` picks a few standard shape measurements that your table has.
- `FEATURE_TRANSFORM`: how measurements are re-expressed before they are coloured: `"rank"` (even spread of colours, a good default for skewed measurements), `"zscore"`, `"log1p"` or `"raw"`.
- `LABELS_TIFF`, `MESHES_DIR`: optional, only for the pictures of typical cells (section 9).

In [ ]:
# ---------------- EDIT ME ----------------
NICHE_EMBEDDING_CSV = "niche_output/niche_embedding.csv"
SHAPE_EMBEDDING_CSV = None     # for example "shape_output/shape_embedding.csv"
METADATA_CSV = None            # for example "metadata.csv"
OUTPUT_DIR = "niche_figure_output"

SHAPE_RESOLUTION = 1.0
NICHE_RESOLUTION = 1.5
GROUP_COL = None               # for example "region"; None = stack_name
COLOR_FEATURES = None          # None = a few standard shape measurements
VIOLIN_FEATURES = None         # None = a few standard shape measurements
FEATURE_TRANSFORM = "rank"     # "rank", "zscore", "log1p" or "raw"
FIGURE_SIZE = "full"           # "half", "full", "a4_landscape" or "a4_portrait"

LABELS_TIFF = None             # optional: labels volume of ONE stack, to draw typical cells
LABELS_STACK_NAME = None       # the stack_name that LABELS_TIFF belongs to (None if the table has one stack)
VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)
MESHES_DIR = None              # optional: <MESHES_DIR>/<stack_name>/cell_<label>.obj
N_GALLERY_CELLS = 5            # typical cells shown per niche cluster

# ---------------- Settings with defaults ----------------
POINT_SIZE = 3
POINT_ALPHA = 0.7
N_MIXING_NEIGHBOURS = 15       # neighbours used in the same-stack fraction (section 10)
N_AGREEMENT_NEIGHBOURS = 50    # neighbours used in the agreement across stacks (section 10)
MAX_CELLS_FOR_MIXING = 50000   # the mixing check uses at most this many cells (random subset)
SAVE_FIGURES = True

## 2. Load the tables

The niche file is the main table. Cells that touch no other cell have no niche
and are left grey or left out of the niche panels.

In [ ]:
out_dir = Path(OUTPUT_DIR)
fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
out_dir.mkdir(parents=True, exist_ok=True)
tab_dir.mkdir(exist_ok=True)

df = h.read_table_csv(NICHE_EMBEDDING_CSV)
if SHAPE_EMBEDDING_CSV:
    shape = h.read_table_csv(SHAPE_EMBEDDING_CSV)
    extra = [c for c in shape.columns if c not in df.columns]
    if extra:
        df = df.merge(shape[["stack_name", "label"] + extra], on=["stack_name", "label"], how="left")
        print(f"Added {len(extra)} column(s) from the shape file.")
df = h.attach_metadata(df, METADATA_CSV)

shape_col = h.res_name(SHAPE_RESOLUTION)
niche_col = h.res_name(NICHE_RESOLUTION, "niche_leiden_r")
for need in ("stack_name", "label", "umap_1", "umap_2", "niche_umap_1", "niche_umap_2", shape_col, niche_col):
    if need not in df.columns:
        have = [c for c in df.columns if "leiden_r" in c]
        raise ValueError(f"The table has no '{need}' column. Cluster columns found: {have}.")

stacks = sorted(df["stack_name"].unique())
has_niche = df[niche_col].notna()
print(f"{len(df):,} cells in {len(stacks)} stack(s); {int(has_niche.sum()):,} have a neighbourhood.")
print(f"{df[shape_col].nunique()} shape clusters, {df[niche_col].nunique()} niche clusters.")

shape_palette = h.assign_palette(df[shape_col])
niche_palette = h.assign_palette(df[niche_col])
niche_pc_cols = sorted([c for c in df.columns if re.fullmatch(r"niche_PC\d+", c)], key=lambda c: int(c[8:]))
shape_pc_cols = sorted([c for c in df.columns if re.fullmatch(r"PC\d+", c)], key=lambda c: int(c[2:]))

DEFAULT_FEATURES = ["sphericity", "true_sphericity", "solidity", "aspect_ratio", "z_elongation",
                    "inertia_elongation", "inertia_flatness", "fill_ratio", "equivalent_diameter", "surface_area"]
color_features = [f for f in (COLOR_FEATURES or DEFAULT_FEATURES) if f in df.columns][:6 if COLOR_FEATURES is None else 99]
violin_features = [f for f in (VIOLIN_FEATURES or DEFAULT_FEATURES) if f in df.columns][:8 if VIOLIN_FEATURES is None else 99]
group_col = GROUP_COL or "stack_name"
print("Measurements coloured:", color_features)
print("Measurements as violins:", violin_features)


def safe(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))


def finish(fig, name):
    """Save the figure (SVG + PNG) and show it."""
    fig.tight_layout()
    if SAVE_FIGURES:
        h.save_fig(fig, name, fig_dir)
    plt.show()

## 3. The niche map coloured by niche cluster

Each dot is a cell, placed by the **shapes around it** (not its own shape).
Cells close together live in similar neighbourhoods. A numbered badge marks the
centre of every niche cluster.

In [ ]:
fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
h.umap_with_medians(df, "niche_umap_1", "niche_umap_2", niche_col, palette=niche_palette, ax=ax, s=POINT_SIZE, prefix="N",
                    title=f"Niche clusters (resolution {NICHE_RESOLUTION})")
ax.set_xlabel("niche UMAP 1")
ax.set_ylabel("niche UMAP 2")
finish(fig, "niche_umap_clusters")

## 4. Shape map and niche map, coloured both ways

Top row: the shape map. Bottom row: the niche map. Left: coloured by shape
cluster. Right: coloured by niche cluster. If a niche is made of one shape
cluster, the colours line up. If a niche mixes several shape clusters, one
niche patch shows several colours.

In [ ]:
w, hgt = h.figure_size(FIGURE_SIZE)
fig, axes = plt.subplots(2, 2, figsize=(w * 1.7, hgt * 1.7))
common = dict(s=POINT_SIZE, alpha=POINT_ALPHA, kind="categorical")
h.umap_scatter(df, "umap_1", "umap_2", shape_col, ax=axes[0, 0], palette=shape_palette, title="Shape map, shape clusters", **common)
h.umap_scatter(df, "umap_1", "umap_2", niche_col, ax=axes[0, 1], palette=niche_palette, title="Shape map, niche clusters", **common)
h.umap_scatter(df, "niche_umap_1", "niche_umap_2", shape_col, ax=axes[1, 0], palette=shape_palette, title="Niche map, shape clusters", **common)
h.umap_scatter(df, "niche_umap_1", "niche_umap_2", niche_col, ax=axes[1, 1], palette=niche_palette, title="Niche map, niche clusters", **common)
finish(fig, "maps_shape_and_niche_cross")

## 5. Maps coloured by a measurement

The same two maps coloured by a shape measurement, so you can see what a region
of the map means in plain terms (for example "round" or "flat").
`FEATURE_TRANSFORM` sets how the numbers become colours. Rank spreads the colours
evenly over the cells, so a few very large cells do not wash out the rest.

In [ ]:
if not color_features:
    print("None of the standard measurements are in your table: set COLOR_FEATURES in the EDIT cell.")
for feat in color_features:
    w, hgt = h.figure_size(FIGURE_SIZE)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(w * 1.7, hgt))
    kw = dict(kind="continuous", transform=FEATURE_TRANSFORM, s=POINT_SIZE, alpha=POINT_ALPHA)
    h.umap_scatter(df, "umap_1", "umap_2", feat, ax=a1, title=f"{feat} on the shape map", **kw)
    h.umap_scatter(df, "niche_umap_1", "niche_umap_2", feat, ax=a2, title=f"{feat} on the niche map", **kw)
    finish(fig, f"maps_coloured_by_{safe(feat)}")

## 6. Violins per niche cluster

For each measurement: how the values spread inside every niche cluster. The
white dot is the median, the black bar the middle half of the cells. Cells with
no niche are left out.

In [ ]:
if violin_features:
    fig = h.violin_grid(df[has_niche], violin_features, niche_col, palette=niche_palette, ncols=4)
    finish(fig, "niche_violins")
else:
    print("No measurements to draw: set VIOLIN_FEATURES in the EDIT cell.")

## 7. Which niches make up each group?

For every group (a value of `GROUP_COL`), the share of its cells in each niche
cluster. Groups can be stacks, regions, treatments, or any column you added.
Each bar adds up to 1.

In [ ]:
if group_col not in df.columns:
    print(f"Column '{group_col}' is not in the table: set GROUP_COL in the EDIT cell.")
else:
    comp = h.composition_table(df, group_col, niche_col)
    comp.to_csv(tab_dir / f"niche_share_by_{safe(group_col)}.csv")
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    h.composition_bar(df, group_col, niche_col, ax=ax, palette=niche_palette, horizontal=comp.shape[0] > 8)
    ax.set_title(f"Niche make-up of each {group_col}", fontsize=h.FONT_TITLE)
    finish(fig, f"niche_share_by_{safe(group_col)}")

## 8. Which shape clusters live in which niches?

Each value is log2(observed / expected). Observed: the share of a shape
cluster's cells that fall in the niche. Expected: the share of all cells in that
niche. 0 = as often as chance, +1 = twice as often, -1 = half as often. A red
square means "this shape is found in this neighbourhood more than chance would
give". A grey square marked "none" is a pair that never occurs. Cells without a niche are left out. The table and the cell counts behind it are saved to `tables/`.

In [ ]:
fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
_, enrich = h.enrichment_heatmap(df[has_niche], shape_col, niche_col, ax=ax)
ax.set_title("Shape cluster by niche cluster", fontsize=h.FONT_TITLE)
enrich.to_csv(tab_dir / "log2_enrichment_shape_by_niche.csv")
pd.crosstab(df.loc[has_niche, shape_col], df.loc[has_niche, niche_col]).to_csv(tab_dir / "cell_counts_shape_by_niche.csv")
finish(fig, "enrichment_shape_by_niche")

if group_col in df.columns and (group_col != "stack_name" or len(stacks) > 1):
    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    _, enrich_g = h.enrichment_heatmap(df[has_niche], group_col, niche_col, ax=ax)
    ax.set_title(f"{group_col} by niche cluster", fontsize=h.FONT_TITLE)
    enrich_g.to_csv(tab_dir / f"log2_enrichment_{safe(group_col)}_by_niche.csv")
    finish(fig, f"enrichment_{safe(group_col)}_by_niche")

## 9. Typical cells of each niche (optional)

For every niche cluster, the cells closest to the cluster's median in **niche**
space: the most typical neighbourhoods. They are listed in
`tables/niche_typical_cells.csv`. The picture shows only the centre cell of a niche; to see its
neighbours, open the stack in the plugin and look for the cell with the stack and label from the table. Pictures need `LABELS_TIFF` (one stack) or
`MESHES_DIR`; without them the picture is skipped.

In [ ]:
if len(niche_pc_cols) >= 2:
    sub = df[has_niche].reset_index(drop=True)
    reps = h.select_representatives(sub[niche_pc_cols].to_numpy(dtype=float), sub, niche_col, N_GALLERY_CELLS)
    reps.to_csv(tab_dir / "niche_typical_cells.csv", index=False)
    print(f"Wrote niche_typical_cells.csv ({len(reps)} rows).")
    if LABELS_TIFF is None and MESHES_DIR is None:
        print("No LABELS_TIFF or MESHES_DIR given: pictures skipped.")
    else:
        labels_volume = h.read_labels_volume(LABELS_TIFF) if LABELS_TIFF else None
        labels_stack = LABELS_STACK_NAME or (stacks[0] if len(stacks) == 1 else None)
        fig = h.plot_cluster_gallery(reps, niche_palette, labels_volume, labels_stack, VOXEL_SIZE_ZYX, MESHES_DIR,
                                     n_per_cluster=N_GALLERY_CELLS)
        if fig is None:
            print("No mesh could be made for the typical cells (check the stack name and the folder).")
        else:
            finish(fig, "niche_typical_cells_gallery")
else:
    print("The file has no niche PC columns, so typical cells cannot be found.")

## 10. Do replicate stacks mix? (several stacks only)

If your table holds several stacks, a good embedding puts similar cells together
**whichever stack they came from**. Two checks, in the shape and the niche PC spaces:

- **Same-stack fraction.** For each cell, the fraction of its nearest neighbours (in PC space, not on the 2D map) that come from its own stack. If stacks mixed perfectly this would equal the value you get after shuffling the stack labels (the "shuffled" bar). Much higher means each stack forms its own island. For the **niche** embedding this fraction is high by construction, because neighbouring cells share most of their neighbourhood: compare it with the shape embedding, and rely on the agreement across stacks below.
- **Agreement across stacks.** Looking only at neighbours from *other* stacks, how often do they share the cell's cluster? `ratio` is observed / expected by chance (1 = no better than chance).

The second table shows how alike two stacks are in their mix of clusters (1 = the same mix).

In [ ]:
if len(stacks) < 2:
    print("One stack in the table: this section is skipped.")
elif len(niche_pc_cols) < 2 or len(shape_pc_cols) < 2:
    print("The file has no PC columns: this section is skipped.")
else:
    mix = df[has_niche]
    if len(mix) > MAX_CELLS_FOR_MIXING:
        mix = mix.sample(MAX_CELLS_FOR_MIXING, random_state=0)
        print(f"Using a random subset of {len(mix):,} cells for this section.")
    rows = []
    for name, cols, cl in (("shape", shape_pc_cols, shape_col), ("niche", niche_pc_cols, niche_col)):
        X = mix[cols].to_numpy(dtype=float)
        score, base = h.stack_mixing_score(X, mix["stack_name"].to_numpy(), k=N_MIXING_NEIGHBOURS)
        rows.append({"embedding": name, "same_stack_fraction": np.nanmean(score), "shuffled_labels": np.nanmean(base)})
        agree = h.cross_group_agreement(X, mix[cl].to_numpy(), mix["stack_name"].to_numpy(), k=N_AGREEMENT_NEIGHBOURS)
        agree.to_csv(tab_dir / f"agreement_across_stacks_{name}.csv")
        print(f"\nAgreement across stacks, {name} clusters (mean ratio {agree['ratio'].mean():.2f}):")
        print(agree.round(3).to_string())
    mixing = pd.DataFrame(rows)
    mixing.to_csv(tab_dir / "stack_mixing.csv", index=False)
    print("\n" + mixing.round(3).to_string(index=False))

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(h.figure_size(FIGURE_SIZE)[0] * 1.7, h.figure_size(FIGURE_SIZE)[1]))
    x = np.arange(len(mixing))
    a1.bar(x - 0.18, mixing["same_stack_fraction"], 0.34, color="#0072B2", label="observed")
    a1.bar(x + 0.18, mixing["shuffled_labels"], 0.34, color="#BBBBBB", label="shuffled stack labels")
    a1.set_xticks(x)
    a1.set_xticklabels(mixing["embedding"])
    a1.set_ylabel("same-stack fraction of neighbours", fontsize=h.FONT_AXIS)
    a1.legend(frameon=False, fontsize=h.FONT_LEGEND)
    overlap = h.sample_overlap(df[has_niche], niche_col)
    h.matrix_heatmap(overlap, ax=a2, vmin=0, vmax=1, cbar_label="overlap of niche mix")
    a2.set_title("Niche mix: stack against stack", fontsize=h.FONT_TITLE)
    finish(fig, "do_replicates_mix")

    fig, ax = plt.subplots(figsize=h.figure_size(FIGURE_SIZE))
    ax, rep_tab = h.replicate_dotplot(df, niche_col, ax=ax, palette=niche_palette)
    rep_tab.to_csv(tab_dir / "niche_share_per_stack.csv", index=False)
    finish(fig, "niche_share_per_stack")

## 11. What was written

Figures (`.svg` editable, `.png` 300 dpi) are in `figures/`; the tables behind
them are in `tables/`.

In [ ]:
for sub in (fig_dir, tab_dir):
    names = sorted(p.name for p in sub.glob("*")) if sub.exists() else []
    print(f"{sub.name}/ ({len(names)} files):")
    for n in names:
        print("   ", n)